# 03 Follow-ups

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['GE_DRIVE_ROOT'] = '/content/drive/MyDrive/hypothesaes_goemotions'
REPO_URL = 'https://github.com/vankadaruvijayberk/HypotheSAEs.git'   # <-- your fork
REPO_DIR = '/content/HypotheSAEs'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Install

In [2]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

!pip install -q -e {REPO_DIR}
!pip install -q -U datasets sentence-transformers transformers scikit-learn statsmodels "numpy<2.0.0"
!pip install -q -U vllm torchaudio torchvision nvidia-cuda-runtime-cu12
!pip uninstall -y torchcodec

# editable-install .pth files are only read at interpreter startup, so add the paths directly
import sys, importlib
for p in (REPO_DIR, os.path.join(REPO_DIR, 'goemotions')):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hypothesaes (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.25.0 requires torchcodec>=0.14, which is not installed.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.25.0 requires torchcodec>=0.14, which is not installed.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is i

## Imports

In [3]:
import numpy as np
from scipy.stats import pearsonr, spearmanr
import ge_pipeline as ge
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.annotate import annotate_texts_with_concepts
from hypothesaes.evaluation import score_hypotheses
ge.set_seed()
ge.redirect_annotation_cache()

## Load Heldout and Hypotheses

In [4]:
ds, NAMES = ge.load_goemotions()
test_texts = list(ds['test']['text'])
test_targets = ge.build_targets(ds['test']['labels'], NAMES)

interp = ge.read_json('06_interpretations')
evaluation = ge.read_json('07_evaluation')
hidx = evaluation['heldout_indices']            # same rows notebook 02 annotated
held_texts = [test_texts[i] for i in hidx]
held_targets = {t: test_targets[t][hidx] for t in test_targets}

selected = {t: [int(n) for n in v] for t, v in interp['selected'].items()}
neuron2hyp = {int(k): v['interpretation'] for k, v in interp['best'].items() if v['interpretation']}
union_hyps = sorted(set(neuron2hyp.values()))
len(held_texts), len(union_hyps)

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

(2000, 64)

## Cosine Annotation

In [5]:
te = get_local_embeddings(held_texts, model=ge.EMBEDDER, nomic_prefix='search_document: ',
                          cache_name='followup_held_doc', batch_size=128)
he = get_local_embeddings(union_hyps, model=ge.EMBEDDER, nomic_prefix='search_query: ',
                          cache_name='followup_hyp_query', batch_size=64)
T = np.stack([te[t] for t in held_texts]).astype(np.float32)
H = np.stack([he[h] for h in union_hyps]).astype(np.float32)
T /= np.linalg.norm(T, axis=1, keepdims=True)
H /= np.linalg.norm(H, axis=1, keepdims=True)
COS = T @ H.T
del te, he, T, H; ge.clear_mem()
COS.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/445k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/1 [00:00<?, ?it/s]

Saved 64 embeddings to /content/drive/MyDrive/hypothesaes_goemotions/emb_cache/followup_hyp_query/chunk_000.npy


(2000, 64)

## Cosine vs LLM Agreement

In [6]:
llm = annotate_texts_with_concepts(texts=held_texts, concepts=union_hyps, model='Qwen/Qwen3-1.7B',
                                   cache_name='goemotions_heldout', n_workers=1,
                                   use_cache_only=True, uncached_value=0, max_words_per_example=64)

agree = {}
for j, h in enumerate(union_hyps):
    a = llm[h].astype(float)
    if a.std() > 0:
        agree[h] = float(pearsonr(COS[:, j], a)[0])
mean_agreement = float(np.mean(list(agree.values()))) if agree else None
ge.log_json('09_cosine_annotation', {'mean_pearson_cosine_vs_llm': mean_agreement,
                                     'per_hypothesis': agree})
mean_agreement

Found 128000 cached items; mapped 0 uncached items to 0


0.17257118369781074

## Predictiveness Comparison

In [7]:
def separation(a, y):
    p, n = a == 1, a == 0
    return float(y[p].mean() - y[n].mean()) if p.sum() and n.sum() else np.nan

llm_sep, cos_sep = [], []
for t, neurons in selected.items():
    y = held_targets[t].astype(float)
    for n in neurons:
        h = neuron2hyp.get(n)
        if not h:
            continue
        j = union_hyps.index(h)
        cos_bin = (COS[:, j] >= np.quantile(COS[:, j], 0.9)).astype(int)
        llm_sep.append(separation(llm[h], y))
        cos_sep.append(separation(cos_bin, y))
llm_sep, cos_sep = np.array(llm_sep), np.array(cos_sep)
mask = ~(np.isnan(llm_sep) | np.isnan(cos_sep))
rho = float(spearmanr(llm_sep[mask], cos_sep[mask]).correlation)
ge.log_json('09_cosine_annotation', {'spearman_predictiveness_llm_vs_cosine': rho})
rho

0.4037567316933376

## Annotator Swap

In [12]:
import subprocess, time, requests
from google.colab import userdata

SWAP_MODELS = ['Qwen/Qwen3-0.6B', 'meta-llama/Llama-3.2-1B-Instruct']
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # <-- required: Llama-3.2-1B-Instruct is gated
except userdata.SecretNotFoundError:
    os.environ['HF_TOKEN'] = ''

if any('meta-llama' in m for m in SWAP_MODELS) and not os.environ.get('HF_TOKEN'):
    raise RuntimeError('set HF_TOKEN, or drop the gated model from SWAP_MODELS')

def serve(model, port=8001):
    log_path = f'/tmp/vllm_{port}.log'
    log_file = open(log_path, 'w')
    p = subprocess.Popen(['vllm', 'serve', model, '--port', str(port),
                          '--max-model-len', '4096', '--gpu-memory-utilization', '0.85'],
                         stdout=log_file, stderr=subprocess.STDOUT)
    base = f'http://127.0.0.1:{port}/v1'
    for _ in range(180):
        try:
            if requests.get(base + '/models', timeout=2).status_code == 200:
                break
        except Exception:
            pass
        time.sleep(5)
    else:
        p.terminate()
        with open(log_path, 'r') as f:
            logs = f.read()
        raise RuntimeError(f'server not up: {model}\n\n--- vLLM LOGS ---\n{logs[-4000:]}')

    os.environ['OPENAI_BASE_URL'] = base
    os.environ.setdefault('OPENAI_API_KEY', 'EMPTY')
    # Attach log_file to process object so we can close it later if needed
    p.log_file = log_file
    return p

swap = {}
for model in SWAP_MODELS:
    proc = serve(model)
    kw = {'temperature': 0.0, 'max_output_tokens': 64}
    if 'Qwen3' in model:   # Llama has no thinking mode; disable Qwen's for a fair comparison
        kw['extra_body'] = {'chat_template_kwargs': {'enable_thinking': False}}
    ann = annotate_texts_with_concepts(texts=held_texts, concepts=union_hyps, model=model,
                                       cache_name='swap_' + model.split('/')[-1],
                                       n_workers=16, max_words_per_example=64, **kw)
    sig = {}
    for t, neurons in selected.items():
        hyps = sorted({neuron2hyp[n] for n in neurons if n in neuron2hyp})
        if len(hyps) < 2:
            continue
        m, _ = score_hypotheses({h: ann[h] for h in hyps},
                                y_true=held_targets[t].astype(float), classification=True)
        sig[t] = m['Significant'][0]
    swap[model] = sig
    proc.terminate()
    if hasattr(proc, 'log_file'):
        proc.log_file.close()
    ge.clear_mem()

ge.log_json('10_annotator_swap', {'significant_by_model': swap})
swap

RuntimeError: server not up: Qwen/Qwen3-0.6B